Business Challenge 1: Application of Customer Sentiment Analysis on Yelp Dataset
Yelp is an online platform where people can rate, review, and share experiences about local
businesses. Customer reviews contain valuable insights into customer satisfaction, product
quality, and service delivery. However, the massive volume makes it impossible for
managers to manually read and interpret all reviews. The company is therefore exploring
automated sentiment analysis to understand customer opinions at scale and make datadriven decisions that improve service quality and competitiveness.

In [ ]:
%pip install datasets
from datasets import load_dataset
dataset = load_dataset("yelp_polarity")
print(dataset)
# 'dataset' is a DatasetDict (not callable). Access splits like this:
print(dataset['train'][0])


In [ ]:
# Imports ( add more if needed )
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import torch
import os
import json




Task I: Understanding Customer Feedback (Exploratory Analysis)
Before building models, the analytics team must understand the structure of customer
feedback:
1. Assess whether there is a class imbalance (e.g., more positive reviews than
negative).
2. Extract and display sample reviews from both positive and negative categories to
illustrate customer voice.
3. Plot the distribution of review lengths to evaluate typical review detail and prepare
for model design.

In [ ]:
# load dataset and preprocess
print(dataset)
train_df = pd.DataFrame(dataset['train'])
#1.1 - Distribution of classes in training set.
# Calculate value counts for the 'label' column
label_counts = train_df['label'].value_counts()

print("--- Class Distribution in Training Set ---")
print(label_counts)

# Calculate percentages
total_samples = len(train_df)
percent_0 = (label_counts[0] / total_samples) * 100
percent_1 = (label_counts[1] / total_samples) * 100

print(f"\nPercentage of Negative (0) reviews: {percent_0:.2f}%")
print(f"Percentage of Positive (1) reviews: {percent_1:.2f}%")

# Visualization of Class Distribution
plt.figure(figsize=(6, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, palette=["red", "green"])
plt.title('Distribution of Yelp Review Sentiment Classes ')
plt.xlabel('Sentiment Label (0: Negative, 1: Positive)')
plt.ylabel('Number of Reviews')
plt.xticks([0, 1], ['Negative', 'Positive'])
plt.show()

In [ ]:
#1.2 - Extract and display sample reviews from both positive and negative categories to illustrate customer voice
# Get sample negative reviews (Label 0)
negative_samples = train_df[train_df['label'] == 0]['text'].head(3)

# Get sample positive reviews (Label 1)
positive_samples = train_df[train_df['label'] == 1]['text'].head(3)

print("--- Sample Negative Reviews (Label 0) ---")
for i, review in enumerate(negative_samples):
    print(f"Review {i+1}:\n{review}\n---")

print("\n--- Sample Positive Reviews (Label 1) ---")
for i, review in enumerate(positive_samples):
    print(f"Review {i+1}:\n{review}\n---")

In [ ]:
#1.3 plot the distribution of review lengths for both positive and negative reviews.

# Create a new column for review length
train_df['review_length'] = train_df['text'].apply(len)

# Plot the distribution of review lengths
plt.figure(figsize=(10, 6))
sns.histplot(train_df['review_length'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of Yelp Review Lengths (in characters) ')
plt.xlabel('Review Length (Characters)')
plt.ylabel('Frequency')
plt.xlim(0, 1000) # Limit x-axis for better visibility of the bulk of the data
plt.show()

# Calculate descriptive statistics for review lengths
print("\n--- Descriptive Statistics for Review Lengths ---")
print(train_df['review_length'].describe())


In [ ]:
# Comparison Plot of Review Lengths by Sentiment
# --- Code for the Comparison Plot ---
plt.figure(figsize=(10, 6))

# Plot the distribution for Negative Reviews (Label 0)
sns.kdeplot(
    train_df[train_df['label'] == 0]['review_length'],
    label='Negative (0)',
    color='red',
    fill=True,
    alpha=0.4,
    linewidth=2
)

# Plot the distribution for Positive Reviews (Label 1)
sns.kdeplot(
    train_df[train_df['label'] == 1]['review_length'],
    label='Positive (1)',
    color='green',
    fill=True,
    alpha=0.4,
    linewidth=2
)

plt.title('Distribution of Review Lengths: Positive vs. Negative')
plt.xlabel('Review Length (Characters)')
plt.ylabel('Density')
# Based on your statistics (75% is 947), limiting to 1000 is still appropriate
plt.xlim(0, 1000) 
plt.legend()
plt.show() # Crucial command to display the plot

TASK 2 - The company wants an initial benchmark model to classify reviews as positive or
negative.
1. Develop a baseline sentiment classifier using TF-IDF + Random Forest.
2. Provide a validation report on a validation partition of the training set showing
precision, recall, F1-score, and support to assess strengths and weaknesses of the
baseline model.
3. Display a confusion matrix on the validation data to illustrate misclassifications.
4. Use grid search optimization to improve the baseline by tuning:
o Number of estimators [100, 500]
o Maximum tree depth [None, 20, 50]
o Minimum samples split [2, 5]
o Minimum samples per leaf [2, 4]
o Maximum features at each node ["sqrt", "log2"]
5. Report the best model parameters and explain why they are important for business
decision-making.

In [ ]:
#2.1 - Develop a Baseline Sentiment Classifier (TF-IDF + Random Forest)
# Use a smaller subset of the training data for faster processing
# Using 50,000 samples for train/validation split
SUBSET_SIZE = 50000
subset_df = train_df.head(SUBSET_SIZE)

X = subset_df['text']
y = subset_df['label']

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

# 1. TF-IDF Vectorization
# Use up to 10,000 most frequent words/n-grams
tfidf_vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

# 2. Random Forest Classifier (Baseline)
# Use default parameters for the baseline
rf_baseline = RandomForestClassifier(random_state=42, n_estimators=100)
rf_baseline.fit(X_train_tfidf, y_train)

# Predict on the validation set
y_pred_baseline = rf_baseline.predict(X_val_tfidf)

In [ ]:
# Baseline Model Validation Report (TF-IDF + Random Forest)
print("\n--- Baseline Model Validation Report (TF-IDF + Random Forest) ---")
print(classification_report(y_val, y_pred_baseline, target_names=['Negative (0)', 'Positive (1)']))

# Display Confusion Matrix
cm = confusion_matrix(y_val, y_pred_baseline)

plt.figure(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative (0)', 'Positive (1)'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Baseline Random Forest Model ')
plt.show()

In [ ]:
# Optimizing the model using Grid Search CV
# Define the parameter grid for tuning
param_grid = {
    'n_estimators': [100, 500],
    'max_depth': [None, 20, 50],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [2, 4],
    'max_features': ["sqrt", "log2"]
}

# Initialize Grid Search with a smaller, more optimized search space
# Using a subset of the training data (X_train_tfidf, y_train)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1), # n_jobs=-1 for parallel processing
    param_grid=param_grid,
    scoring='f1', # Optimize for F1-score
    cv=3,        # 3-fold cross-validation
    verbose=1,
    n_jobs=-1
)

# Fit the grid search (This can take several minutes)
print("\n--- Starting Grid Search Optimization (This may take time) ---")
grid_search.fit(X_train_tfidf, y_train)

# Get the best parameters and the corresponding score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

In [ ]:
print(f"\n--- Best Random Forest Hyperparameters (Grid Search) ---")
print(best_params)
print(f"Best Cross-Validation F1 Score: {best_score:.4f}")

# Train the best model and evaluate on the validation set
rf_optimized = grid_search.best_estimator_
y_pred_optimized = rf_optimized.predict(X_val_tfidf)

print("\n--- Optimized Model Validation Report ---")
print(classification_report(y_val, y_pred_optimized, target_names=['Negative (0)', 'Positive (1)']))

In [ ]:
# Display Confusion Matrix for Optimized Model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Get the best performing model from Grid Search
rf_optimized = grid_search.best_estimator_

# 2. Predict on the validation set using the optimized model
# X_val_tfidf is the validation data transformed by the TF-IDF vectorizer
y_pred_optimized = rf_optimized.predict(X_val_tfidf)

# 3. Calculate the Confusion Matrix
# y_val is the true sentiment labels for the validation set
cm_optimized = confusion_matrix(y_val, y_pred_optimized)

# 4. Display the Confusion Matrix
plt.figure(figsize=(7, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_optimized,
    display_labels=['Negative (0)', 'Positive (1)']
)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Confusion Matrix for Optimized Random Forest Model ')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# 5. Print the Classification Report for a detailed view
print("\n--- Optimized Random Forest Model Validation Report ---")
print(classification_report(y_val, y_pred_optimized, target_names=['Negative (0)', 'Positive (1)']))

Task III: Advanced Sentiment Intelligence with Pretrained Models
The company’s leadership is interested in how cutting-edge AI could outperform the
baseline.
1. Implement two transformer-based pretrained models (from Hugging Face) for
sentiment classification and discuss why you selected each model.
2. Compare their performance against each other and against the optimized Random
Forest model.
3. Quantify how much better the transformer models are and explain the business
impact (e.g., fewer false negatives could prevent overlooking unhappy customers,
better precision could reduce escalation costs).

Model	Justification
1. bert-base-uncased (Base Model)	Industry Standard Baseline: BERT is the foundational transformer model. It's a great choice for a first advanced benchmark, as its architecture is well-understood, and it provides a strong performance baseline before moving to larger or more specialized models.
2. roberta-base (Improved Model)	Performance Leader: RoBERTa (Robustly Optimized BERT Pretraining Approach) is an optimized version of BERT. It was trained longer, on more data, and with different pre-training objectives (dynamic masking), often leading to better performance on downstream tasks like sentiment classification. This showcases the benefit of using state-of-the-art techniques.

In [10]:
!pip install -U transformers==4.35.2 torch accelerate

   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.9 MB 5.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/7.9 MB 5.3 MB/s eta 0:00:02
   ----------------- ---------------------- 3.4/7.9 MB 5.6 MB/s eta 0:00:01
   ---------------------- ----------------- 4.5/7.9 MB 5.5 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/7.9 MB 5.5 MB/s eta 0:00:01
   ---------------------------------- ----- 6.8/7.9 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------  7.9/7.9 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 7.9/7.9 MB 5.3 MB/s  0:00:01
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 5.6 MB/s eta 0:00:01
   -------------------------------------- - 2.1/2.2 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 5.1 MB/s  0:00:00
   -----------------------------

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Define the Model Checkpoints (from Hugging Face Hub)
ROBERTA_MODEL = "siebert/sentiment-roberta-large-english"
DISTILBERT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# 1. Load RoBERTa (High Performance Model)
print(f"Loading RoBERTa: {ROBERTA_MODEL}...")
tokenizer_roberta = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model_roberta = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL)
print("✅ RoBERTa model loaded successfully!")

# 2. Load DistilBERT (High Efficiency Model)
print(f"\nLoading DistilBERT: {DISTILBERT_MODEL}...")
tokenizer_distilbert = AutoTokenizer.from_pretrained(DISTILBERT_MODEL)
model_distilbert = AutoModelForSequenceClassification.from_pretrained(DISTILBERT_MODEL)
print("✅ DistilBERT model loaded successfully!")

# Final check: Print the type of one loaded model
print(f"\nLoaded RoBERTa object type: {type(model_roberta)}")

c:\Users\fabio\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\fabio\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Loading RoBERTa: siebert/sentiment-roberta-large-english...


c:\Users\fabio\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\fabio\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fabio\.cache\huggingface\hub\models--siebert--sentiment-roberta-large-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either n

✅ RoBERTa model loaded successfully!

Loading DistilBERT: distilbert/distilbert-base-uncased-finetuned-sst-2-english...


c:\Users\fabio\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\fabio\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fabio\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Wi

✅ DistilBERT model loaded successfully!

Loaded RoBERTa object type: <class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'>


In [2]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import torch # Already loaded, but good practice to keep here

In [ ]:
# 1. Prepare the Validation Data for Transformer Comparison
# Use the same variable names as previously defined for train/validation split
df_validation = pd.DataFrame({
    'text': X_val.tolist(),
    'label': y_val.tolist()
})

# 2. Get the Optimized Random Forest Predictions (The Baseline)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
df_validation['rf_prediction'] = rf_optimized.predict(X_val_tfidf)

print(f"Validation DataFrame ready with {len(df_validation)} samples.")
print("Baseline RF predictions (rf_prediction) are included.")

NameError: name 'X_val' is not defined

Business Challenge 2: Books recommendation system development
Digital book platform (similar to Goodreads or Amazon Kindle) always faces the business
challenge of how identify similar books from as well as how books can be recommended to
readers that they are most likely to enjoy, based on their past ratings and the behavior of
other users.
The Goodbooks-10k dataset available in this link, is a popular dataset used for
benchmarking recommender systems. The dataset contains customer rating on thousands
of books. The file dataset can be briefly described as follows:
• user_id: An anonymized identifier for each reader.
• book_id: An identifier for each book in the collection (covering 10,000 unique
books).
• rating: A rating given by the user to the book, ranging from 1 (lowest) to 5 (highest).
• Over 6 million ratings provided by more than 50,000 users.
• Most users have rated only a handful of books, and most books are rated by only a
subset of users.
Using this dataset conduct recommendation system analysis by following the tasks
given below
Task I: Exploratory Data Analysis (EDA)
Perform exploratory data analysis on the provided dataset.
- Summarize the dataset: number of rows, columns, and types of variables.
- Show the distribution of labels or ratings.
- Provide at least two visualizations (e.g., histogram, bar chart, word cloud).

In [ ]:
#Imports for the Business Challenge 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares # pyright: ignore[reportMissingImports]
import warnings
warnings.filterwarnings('ignore') # Suppress warnings for cleaner notebook

In [ ]:
#Task 1  - LOAD DATASET 

# Define the dataset URL
DATA_URL = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"

# Load the data directly from the URL
print("Loading data...")
ratings_df = pd.read_csv(DATA_URL)
print(f"Original ratings loaded: {len(ratings_df)}")
print("\nInitial Data Sample:")
print(ratings_df.head())
# --- Data Preparation for 'implicit' ---

# The 'implicit' library works best with user and item IDs
# that are contiguous integers starting from zero.

# 3. Create mapping from original IDs to new contiguous IDs
def create_mappings(df, column_name):
    """Maps original IDs in a column to a contiguous integer range."""
    # Create a Series mapping original ID to the new 0-indexed ID
    id_to_index = {
        original_id: index
        for index, original_id in enumerate(df[column_name].unique())
    }
    # Create the inverse mapping (new index to original ID)
    index_to_id = {index: original_id for original_id, index in id_to_index.items()}
    return id_to_index, index_to_id

# Map user_id and book_id
user_to_index, index_to_user = create_mappings(ratings_df, 'user_id')
book_to_index, index_to_book = create_mappings(ratings_df, 'book_id')

# Apply the mapping to the DataFrame
ratings_df['user_index'] = ratings_df['user_id'].apply(lambda x: user_to_index[x])
ratings_df['book_index'] = ratings_df['book_id'].apply(lambda x: book_to_index[x])

# 4. Create the Sparse Matrix (Item-User format)
# The ALS model in 'implicit' trains faster using the Item-User matrix, 
# where items are rows and users are columns.
# It expects the data to be in "confidence" format, where higher values indicate
# stronger preference (your ratings are fine for this).

print("Creating sparse matrix...")
item_user_data = csr_matrix(
    (
        ratings_df['rating'].astype(float),  # The values (ratings/confidence)
        (
            ratings_df['book_index'],       # The row indices (Items)
            ratings_df['user_index']        # The column indices (Users)
        )
    ),
    shape=(len(book_to_index), len(user_to_index))
)

print(f"Data ready! Sparse Matrix shape: {item_user_data.shape} (Items x Users)")

In [ ]:
## #Task 1 EDA: Rating Distribution (Optional but good check)

plt.figure(figsize=(7, 5))
sns.countplot(x='rating', data=ratings_df, palette='viridis')
plt.title('Distribution of User Ratings (1-5)')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

## B. Data Preprocessing & Filtering

# 1. Filter out inactive users (Users who rated fewer than 50 books)
min_user_ratings = 50
user_counts = ratings_df['user_id'].value_counts()
active_users = user_counts[user_counts >= min_user_ratings].index
ratings_filtered = ratings_df[ratings_df['user_id'].isin(active_users)].copy()
print(f"\nRatings after filtering inactive users (< {min_user_ratings} ratings): {len(ratings_filtered)}")

# 2. Filter out unpopular books (Books with fewer than 50 ratings)
min_book_ratings = 50
book_counts = ratings_filtered['book_id'].value_counts()
popular_books = book_counts[book_counts >= min_book_ratings].index
ratings_filtered = ratings_filtered[ratings_filtered['book_id'].isin(popular_books)].copy()
print(f"Final filtered ratings for active users/popular books: {len(ratings_filtered)}")

# 2. Create the histogram
plt.figure(figsize=(10, 6))

# Plot the distribution of the counts
plt.hist(
    user_counts, 
    bins=50, 
    color='teal', 
    edgecolor='black',
    log=True # Use a log scale on the y-axis to see the high-frequency bins clearly
)

plt.title('Distribution of Ratings Per User (Filtered Data)')
plt.xlabel(f'Number of Books Rated (Min Rated: {book_counts.min()})')
plt.ylabel('Number of Users (Log Scale)')
plt.grid(axis='y', alpha=0.5)
plt.show()

Task II: Apply the collaborative filtering model, Alternating Least Squares (ALS).
The model is available in the implicit python library. You may use “pip install implicit” to
install the library.
a) Explain the ALS model in your own words.
- What problem does it solve?
- How does it use matrix factorization to recommend items?
- What are its main hyperparameters?
b) Train the ALS model on a training set only.
- Convert the dataset into a user–item interaction matrix.
- Fit the ALS model using the implicit library.
- Report factors, regularization, iterations, and any other key parameters.

In [ ]:
## D. Matrix Preparation for iALS

# 1. Re-index User and Book IDs to be contiguous starting from 0
ratings_filtered['user_index'] = ratings_filtered['user_id'].astype('category').cat.codes
ratings_filtered['book_index'] = ratings_filtered['book_id'].astype('category').cat.codes

# Create mapping dictionaries to go back to original IDs
user_id_to_index = dict(zip(ratings_filtered['user_id'], ratings_filtered['user_index']))
book_id_to_index = dict(zip(ratings_filtered['book_id'], ratings_filtered['book_index']))
index_to_book_id = dict(zip(ratings_filtered['book_index'], ratings_filtered['book_id']))

# 2. Create the Item-User Sparse Matrix (Required by implicit.als)
# The matrix should be of shape (num_items, num_users)
num_users = ratings_filtered['user_index'].nunique()
num_books = ratings_filtered['book_index'].nunique()

# Use the rating value to represent confidence (or simply preference, as it's explicit data)
# NOTE: The 'implicit' library typically expects implicit feedback (e.g., 1 for interaction, 0 for none).
# For explicit ratings (1-5), we can use the rating itself or a binarized version. We'll use the rating as confidence.
item_user_matrix = csr_matrix((
    ratings_filtered['rating'].astype(np.float32), 
    (ratings_filtered['book_index'], ratings_filtered['user_index'])
), shape=(num_books, num_users))

print(f"\nItem-User Matrix Shape: {item_user_matrix.shape}")

In [ ]:
## E. Model Training: iALS

# Initialize the ALS model
model = AlternatingLeastSquares(
    factors=64,             # Number of latent factors (dimensionality of embeddings)
    regularization=0.05,    # Regularization strength to prevent overfitting
    alpha=2.0,              # Confidence level multiplier (used for implicit data, set higher for explicit ratings)
    iterations=20           # Number of optimization iterations
)

# Train the model (using the Item-User matrix)
print("\nStarting iALS Model Training...")
# NOTE: The .fit() method expects an Item-User matrix.
model.fit(item_user_matrix)
print("Model training complete.")

# Assuming your model training is complete: model.fit(item_user_matrix)

# 1. Access the learned item factors (Book Embeddings)
item_factors = model.item_factors
print("\n--- Item Factors (Book Embeddings) ---")
print(f"Shape: {item_factors.shape}")
print(f"Sample Row (First Book): {item_factors[0][:5]}...") # Prints the first 5 dimensions of the first book's factor vector

# 2. Access the learned user factors (User Preference Vectors)
user_factors = model.user_factors
print("\n--- User Factors (Preference Vectors) ---")
print(f"Shape: {user_factors.shape}")
print(f"Sample Row (First User): {user_factors[0][:5]}...") # Prints the first 5 dimensions of the first user's factor vector

In [ ]:
## 5. Generating and Interpreting Recommendations

# Load book metadata (titles) if not already available
# (Books CSV is small; safe to load here for display purposes)
books_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"
try:
    books_df
except NameError:
    books_df = pd.read_csv(books_url, usecols=['book_id', 'title'])

# Function to get the title of a book index (uses index_to_book mapping)
def get_book_title(book_index):
    # index_to_book maps local book_index -> original book_id
    original_book_id = index_to_book.get(book_index) or index_to_book_id.get(book_index)
    if original_book_id is None:
        return "Unknown Book"
    row = books_df[books_df['book_id'] == original_book_id]
    return row['title'].iloc[0] if not row.empty else "Unknown Book"

# 1. Select a Test User (e.g., the first active user)
test_user_id = ratings_filtered['user_id'].iloc[0]
test_user_index = user_id_to_index[test_user_id]
N = 10  # Number of recommendations

# 2. Prepare user_items matrix ensuring it has the correct shape expected by the model.
# The recommend() function expects a user x item matrix (rows = users).
from scipy.sparse import vstack, hstack, csr_matrix

# Desired shapes from the trained model
num_model_users = model.user_factors.shape[0]
num_model_items = model.item_factors.shape[0]

# item_user_matrix is (num_items, num_users)
# Transpose to get user x item matrix
user_items_full = item_user_matrix.T.tocsr()  # shape should be (num_users, num_items)

# If the transposed matrix already matches the model's expected shape, use it directly.
if user_items_full.shape == (num_model_users, num_model_items):
    user_items = user_items_full
else:
    # Otherwise, create a matrix of the expected shape and copy the overlapping part.
    # This handles cases where the matrix used to train the model has different indexing ranges.
    rows_to_copy = min(user_items_full.shape[0], num_model_users)
    cols_to_copy = min(user_items_full.shape[1], num_model_items)

    # Copy the overlapping submatrix
    sub = user_items_full[:rows_to_copy, :cols_to_copy].tocsr()

    # Pad rows if needed
    if rows_to_copy < num_model_users:
        pad_rows = num_model_users - rows_to_copy
        pad_row_block = csr_matrix((pad_rows, cols_to_copy), dtype=user_items_full.dtype)
        top = vstack([sub, pad_row_block]).tocsr()
    else:
        top = sub

    # Pad columns if needed
    if cols_to_copy < num_model_items:
        pad_cols = num_model_items - cols_to_copy
        pad_col_block = csr_matrix((num_model_users, pad_cols), dtype=user_items_full.dtype)
        user_items = hstack([top, pad_col_block]).tocsr()
    else:
        user_items = top

# Final sanity check: ensure the test user index is within bounds of user_items
if test_user_index >= user_items.shape[0] or test_user_index < 0:
    raise ValueError(
        f"test_user_index ({test_user_index}) is out of bounds for user_items with {user_items.shape[0]} rows. "
        "Verify that test_user_index was computed using the same re-indexing/mapping as the matrix used by the model."
    )

# 3. Generate Recommendations
# Now call recommend with the aligned user_items matrix
recommendations, scores = model.recommend(
    userid=test_user_index,
    user_items=user_items,
    N=N,
    filter_already_liked_items=True
)

# 4. Compile and Display Results
recommended_books = []
for book_index, score in zip(recommendations, scores):
    title = get_book_title(book_index)
    recommended_books.append({
        'Book Title': title,
        'Predicted Score': f"{score:.4f}"
    })

print(f"\n--- Top {N} Recommendations for User ID {test_user_id} (index {test_user_index}) ---")
print(pd.DataFrame(recommended_books))

# 5. Display Books Already Rated by the User for context
rated_books = ratings_filtered[ratings_filtered['user_id'] == test_user_id]
# Merge with books_df (has original book_id)
rated_books_info = pd.merge(rated_books, books_df, on='book_id', how='left')
print(f"\n--- Top 5 Books User ID {test_user_id} Has Already Rated ---")
print(rated_books_info[['title', 'rating']].sort_values(by='rating', ascending=False).head(5)) 5. Generating and Interpreting Recommendations

# Load book metadata (titles) if not already available
# (Books CSV is small; safe to load here for display purposes)
books_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"
try:
    books_df
except NameError:
    books_df = pd.read_csv(books_url, usecols=['book_id', 'title'])

# Function to get the title of a book index (uses index_to_book mapping)
def get_book_title(book_index):
    # index_to_book maps local book_index -> original book_id
    original_book_id = index_to_book.get(book_index) or index_to_book_id.get(book_index)
    if original_book_id is None:
        return "Unknown Book"
    row = books_df[books_df['book_id'] == original_book_id]
    return row['title'].iloc[0] if not row.empty else "Unknown Book"

# 1. Select a Test User (e.g., the first active user)
test_user_id = ratings_filtered['user_id'].iloc[0]
test_user_index = user_id_to_index[test_user_id]
N = 10  # Number of recommendations

# 2. Prepare user_items matrix ensuring it has the correct shape expected by the model.
# The recommend() function expects a user x item matrix (rows = users).
from scipy.sparse import vstack, hstack, csr_matrix

# Desired shapes from the trained model
num_model_users = model.user_factors.shape[0]
num_model_items = model.item_factors.shape[0]

# item_user_matrix is (num_items, num_users)
# Transpose to get user x item matrix
user_items_full = item_user_matrix.T.tocsr()  # shape should be (num_users, num_items)

# If the transposed matrix already matches the model's expected shape, use it directly.
if user_items_full.shape == (num_model_users, num_model_items):
    user_items = user_items_full
else:
    # Otherwise, create a matrix of the expected shape and copy the overlapping part.
    # This handles cases where the matrix used to train the model has different indexing ranges.
    rows_to_copy = min(user_items_full.shape[0], num_model_users)
    cols_to_copy = min(user_items_full.shape[1], num_model_items)

    # Copy the overlapping submatrix
    sub = user_items_full[:rows_to_copy, :cols_to_copy].tocsr()

    # Pad rows if needed
    if rows_to_copy < num_model_users:
        pad_rows = num_model_users - rows_to_copy
        pad_row_block = csr_matrix((pad_rows, cols_to_copy), dtype=user_items_full.dtype)
        top = vstack([sub, pad_row_block]).tocsr()
    else:
        top = sub

    # Pad columns if needed
    if cols_to_copy < num_model_items:
        pad_cols = num_model_items - cols_to_copy
        pad_col_block = csr_matrix((num_model_users, pad_cols), dtype=user_items_full.dtype)
        user_items = hstack([top, pad_col_block]).tocsr()
    else:
        user_items = top

# Final sanity check: ensure the test user index is within bounds of user_items
if test_user_index >= user_items.shape[0] or test_user_index < 0:
    raise ValueError(
        f"test_user_index ({test_user_index}) is out of bounds for user_items with {user_items.shape[0]} rows. "
        "Verify that test_user_index was computed using the same re-indexing/mapping as the matrix used by the model."
    )

# 3. Generate Recommendations
# Now call recommend with the aligned user_items matrix
recommendations, scores = model.recommend(
    userid=test_user_index,
    user_items=user_items,
    N=N,
    filter_already_liked_items=True
)

# 4. Compile and Display Results
recommended_books = []
for book_index, score in zip(recommendations, scores):
    title = get_book_title(book_index)
    recommended_books.append({
        'Book Title': title,
        'Predicted Score': f"{score:.4f}"
    })

print(f"\n--- Top {N} Recommendations for User ID {test_user_id} (index {test_user_index}) ---")
print(pd.DataFrame(recommended_books))

# 5. Display Books Already Rated by the User for context
rated_books = ratings_filtered[ratings_filtered['user_id'] == test_user_id]
# Merge with books_df (has original book_id)
rated_books_info = pd.merge(rated_books, books_df, on='book_id', how='left')
print(f"\n--- Top 5 Books User ID {test_user_id} Has Already Rated ---")
print(rated_books_info[['title', 'rating']].sort_values(by='rating', ascending=False).head(5))